# Part 2: Polynomial Regression for Stellar Luminosity
## Modeling L as a function of M and T using Polynomial Features

In this notebook, we implement **polynomial regression from first principles** to model the relationship between stellar mass (M), temperature (T), and luminosity (L):

$$\hat{L} = X \mathbf{w} + b$$

where the feature matrix X includes:
- M (mass)
- T (temperature)
- M² (quadratic mass)
- M·T (interaction term)

We will:
1. Visualize the dataset with two features
2. Engineer polynomial features
3. Implement vectorized loss and gradients
4. Compare multiple feature selection models
5. Analyze interaction effects
6. Perform inference on new data

## 1. Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configure matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)

# Define the dataset (two features)
M = np.array([0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4])  # stellar mass (solar masses)
T = np.array([3800, 4400, 5800, 6400, 6900, 7400, 7900, 8300, 8800, 9200])  # temperature (K)
L = np.array([0.15, 0.35, 1.00, 2.30, 4.10, 7.00, 11.2, 17.5, 25.0, 35.0])  # luminosity (solar luminosities)
n_samples = len(M)

print(f"Dataset: {n_samples} stellar samples")
print(f"Mass range: {M.min():.1f} - {M.max():.1f} M☉")
print(f"Temperature range: {T.min():.0f} - {T.max():.0f} K")
print(f"Luminosity range: {L.min():.2f} - {L.max():.1f} L☉")

## 2. Dataset Visualization with Two Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: M vs L with T encoded by color
ax = axes[0]
scatter = ax.scatter(M, L, c=T, s=150, cmap='coolwarm', edgecolors='black', linewidth=1.5, alpha=0.8)
ax.set_xlabel('Mass (M☉)', fontsize=11, fontweight='bold')
ax.set_ylabel('Luminosity (L☉)', fontsize=11, fontweight='bold')
ax.set_title('M vs L (color = Temperature)', fontsize=12, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Temperature (K)', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 2: T vs L
ax = axes[1]
ax.scatter(T, L, s=150, alpha=0.7, edgecolors='black', linewidth=1.5, c='orange')
ax.set_xlabel('Temperature (K)', fontsize=11, fontweight='bold')
ax.set_ylabel('Luminosity (L☉)', fontsize=11, fontweight='bold')
ax.set_title('T vs L', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 3: Correlation
ax = axes[2]
data_matrix = np.column_stack([M, T, L])
corr = np.corrcoef(data_matrix.T)
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks([0, 1, 2])
ax.set_yticks([0, 1, 2])
ax.set_xticklabels(['M', 'T', 'L'])
ax.set_yticklabels(['M', 'T', 'L'])
ax.set_title('Correlation Matrix', fontsize=12, fontweight='bold')
# Add text annotations
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center', color='black', fontweight='bold')
plt.colorbar(im, ax=ax, label='Correlation')

plt.tight_layout()
plt.show()

print("\n=== Data Interpretation ===")
print(f"Correlation(M, L) = {np.corrcoef(M, L)[0, 1]:.4f} (strong positive)")
print(f"Correlation(T, L) = {np.corrcoef(T, L)[0, 1]:.4f} (strong positive)")
print(f"Correlation(M, T) = {np.corrcoef(M, T)[0, 1]:.4f} (strong positive)")
print("\nBoth M and T contribute to L, and they are correlated themselves.")

## 3. Feature Engineering: Build Polynomial Feature Matrix

In [ ]:
def build_feature_matrix(M, T, feature_set='full'):
    """
    Build feature matrix X with specified features.
    
    Parameters:
    -----------
    M, T : np.ndarray
        Input features
    feature_set : str
        'linear': [M, T]
        'quadratic': [M, T, M²]
        'full': [M, T, M², M·T]
    
    Returns:
    --------
    X : np.ndarray (n_samples, n_features)
        Feature matrix
    """
    n = len(M)
    
    if feature_set == 'linear':
        X = np.column_stack([M, T])
    elif feature_set == 'quadratic':
        X = np.column_stack([M, T, M**2])
    elif feature_set == 'full':
        X = np.column_stack([M, T, M**2, M*T])
    else:
        raise ValueError(f"Unknown feature_set: {feature_set}")
    
    return X

# Build feature matrices
X_linear = build_feature_matrix(M, T, 'linear')
X_quadratic = build_feature_matrix(M, T, 'quadratic')
X_full = build_feature_matrix(M, T, 'full')

print("Feature matrices shapes:")
print(f"  Linear [M, T]: {X_linear.shape}")
print(f"  Quadratic [M, T, M²]: {X_quadratic.shape}")
print(f"  Full [M, T, M², M·T]: {X_full.shape}")

print("\nFirst 5 samples (Full feature matrix):")
print("    M      T       M²        M·T")
for i in range(5):
    print(f"{X_full[i, 0]:.1f}  {X_full[i, 1]:.0f}   {X_full[i, 2]:.2f}  {X_full[i, 3]:.0f}")

## 4. Implement Vectorized Loss and Gradients

**Hypothesis:**
$$\hat{L} = X \mathbf{w} + b$$

**Vectorized MSE loss:**
$$J(\mathbf{w}, b) = \frac{1}{n} \| X \mathbf{w} + b - L \|^2$$

**Gradients:**
$$\frac{\partial J}{\partial \mathbf{w}} = \frac{2}{n} X^T (X \mathbf{w} + b - L)$$
$$\frac{\partial J}{\partial b} = \frac{2}{n} \sum (X \mathbf{w} + b - L)$$

In [ ]:
def predict_poly(X, w, b):
    """
    Vectorized prediction for polynomial regression.
    
    Parameters:
    -----------
    X : np.ndarray (n_samples, n_features)
        Feature matrix
    w : np.ndarray (n_features,)
        Weights
    b : float
        Bias
    
    Returns:
    --------
    L_hat : np.ndarray (n_samples,)
        Predictions
    """
    return X @ w + b

def compute_mse_poly(X, L, w, b):
    """
    Compute MSE loss for polynomial regression.
    """
    L_hat = predict_poly(X, w, b)
    residuals = L_hat - L
    J = np.mean(residuals ** 2)
    return J

def compute_gradients_poly(X, L, w, b):
    """
    Compute gradients for polynomial regression (vectorized).
    
    Returns:
    --------
    dJ_dw : np.ndarray (n_features,)
        Gradient w.r.t. weights
    dJ_db : float
        Gradient w.r.t. bias
    """
    n = len(L)
    L_hat = predict_poly(X, w, b)
    residuals = L_hat - L
    
    dJ_dw = 2.0 * X.T @ residuals / n
    dJ_db = 2.0 * np.sum(residuals) / n
    
    return dJ_dw, dJ_db

# Test with full feature matrix
n_features = X_full.shape[1]
w_test = np.ones(n_features) * 0.1
b_test = 0.1

J_test = compute_mse_poly(X_full, L, w_test, b_test)
dJ_dw_test, dJ_db_test = compute_gradients_poly(X_full, L, w_test, b_test)

print(f"Test with w = {w_test}, b = {b_test}:")
print(f"  J = {J_test:.6f}")
print(f"  dJ/dw = {dJ_dw_test}")
print(f"  dJ/db = {dJ_db_test:.6f}")

## 5. Implement Gradient Descent for Polynomial Regression

In [ ]:
def gradient_descent_poly(X, L, learning_rate=0.0001, n_iterations=2000, verbose=False):
    """
    Gradient descent for polynomial regression.
    
    Returns:
    --------
    w, b, history : tuple
        Final parameters and loss history
    """
    n_features = X.shape[1]
    w = np.zeros(n_features)
    b = 0.0
    history = []
    
    for iteration in range(n_iterations):
        # Compute gradients
        dJ_dw, dJ_db = compute_gradients_poly(X, L, w, b)
        
        # Update parameters
        w = w - learning_rate * dJ_dw
        b = b - learning_rate * dJ_db
        
        # Compute loss
        J = compute_mse_poly(X, L, w, b)
        history.append(J)
        
        if verbose and (iteration % 200 == 0):
            print(f"Iter {iteration}: J={J:.6f}")
    
    return w, b, np.array(history)

# Train the full model
print("Training FULL model [M, T, M², M·T]...")
w_full, b_full, hist_full = gradient_descent_poly(
    X_full, L, learning_rate=0.00001, n_iterations=2000, verbose=True
)
print(f"\nFinal parameters (Full model):")
for i, feat_name in enumerate(['M', 'T', 'M²', 'M·T']):
    print(f"  w[{i}] ({feat_name}): {w_full[i]:.8f}")
print(f"  b (intercept): {b_full:.8f}")
print(f"  Final loss: {hist_full[-1]:.8f}")

In [ ]:
# Plot convergence for full model
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(hist_full, linewidth=2, color='blue')
plt.xlabel('Iteration', fontsize=11, fontweight='bold')
plt.ylabel('Loss J(w, b)', fontsize=11, fontweight='bold')
plt.title('Full Model Convergence (Linear Scale)', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.semilogy(hist_full, linewidth=2, color='blue')
plt.xlabel('Iteration', fontsize=11, fontweight='bold')
plt.ylabel('Loss J(w, b)', fontsize=11, fontweight='bold')
plt.title('Full Model Convergence (Log Scale)', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"Convergence summary:")
print(f"  Initial loss: {hist_full[0]:.6f}")
print(f"  Final loss: {hist_full[-1]:.6f}")
print(f"  Reduction: {(1 - hist_full[-1]/hist_full[0])*100:.2f}%")

## 6. Feature Selection: Compare Three Models

In [ ]:
# Train all three models
models = {}

feature_sets = [
    ('linear', 'M1: [M, T]', X_linear),
    ('quadratic', 'M2: [M, T, M²]', X_quadratic),
    ('full', 'M3: [M, T, M², M·T]', X_full)
]

for feature_set, label, X in feature_sets:
    print(f"\nTraining {label}...")
    w, b, history = gradient_descent_poly(
        X, L, learning_rate=0.00001, n_iterations=2000, verbose=False
    )
    J_final = history[-1]
    models[label] = {'w': w, 'b': b, 'loss': J_final, 'history': history, 'X': X}
    print(f"  Final loss: {J_final:.8f}")
    print(f"  Parameters: {w}")

# Summary table
print("\n" + "="*80)
print("FEATURE SELECTION SUMMARY")
print("="*80)
print(f"{'Model':<25} {'n_features':<12} {'Final Loss':<20}")
print("-"*80)
for label in models.keys():
    n_feats = len(models[label]['w'])
    loss = models[label]['loss']
    print(f"{label:<25} {n_feats:<12} {loss:<20.8f}")
print("="*80)

In [ ]:
# Plot convergence comparison
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
for label in models.keys():
    plt.plot(models[label]['history'], label=label, linewidth=2)
plt.xlabel('Iteration', fontsize=11, fontweight='bold')
plt.ylabel('Loss J(w, b)', fontsize=11, fontweight='bold')
plt.title('Feature Selection: Convergence Comparison', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for label in models.keys():
    plt.semilogy(models[label]['history'], label=label, linewidth=2)
plt.xlabel('Iteration', fontsize=11, fontweight='bold')
plt.ylabel('Loss J(w, b)', fontsize=11, fontweight='bold')
plt.title('Feature Selection: Convergence (Log Scale)', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (label, model_data) in enumerate(models.items()):
    ax = axes[idx]
    
    X = model_data['X']
    w = model_data['w']
    b = model_data['b']
    
    # Compute predictions
    L_hat = predict_poly(X, w, b)
    residuals = L_hat - L
    
    # Plot
    ax.scatter(L, L_hat, s=100, alpha=0.7, edgecolors='black', linewidth=1.5)
    # Perfect prediction line
    L_min, L_max = L.min(), L.max()
    ax.plot([L_min, L_max], [L_min, L_max], 'r--', linewidth=2, label='Perfect fit')
    
    # Add loss info
    rmse = np.sqrt(model_data['loss'])
    ax.set_xlabel('Actual Luminosity (L☉)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted Luminosity (L☉)', fontsize=11, fontweight='bold')
    ax.set_title(f'{label}\nRMSE={rmse:.6f}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Model Comparison ===")
for label in models.keys():
    X = models[label]['X']
    w = models[label]['w']
    b = models[label]['b']
    L_hat = predict_poly(X, w, b)
    residuals = L_hat - L
    rmse = np.sqrt(np.mean(residuals**2))
    print(f"\n{label}:")
    print(f"  RMSE: {rmse:.6f}")
    print(f"  Max |residual|: {np.max(np.abs(residuals)):.6f}")

## 7. Analyze Interaction Effects

For the full model, vary the interaction coefficient $w_{M·T}$ while keeping other parameters fixed.

In [ ]:
# Fix M, T, M² weights; vary only M·T weight
w_M = w_full[0]
w_T = w_full[1]
w_M2 = w_full[2]
w_MT_base = w_full[3]
b_full_val = b_full

# Create a range of M·T weights
w_MT_range = np.linspace(-5e-5, 5e-5, 50)
cost_values = []

for w_MT in w_MT_range:
    # Build modified weight vector
    w_modified = np.array([w_M, w_T, w_M2, w_MT])
    
    # Compute cost
    J = compute_mse_poly(X_full, L, w_modified, b_full_val)
    cost_values.append(J)

cost_values = np.array(cost_values)

# Find minimum
min_idx = np.argmin(cost_values)
w_MT_optimal = w_MT_range[min_idx]
J_min = cost_values[min_idx]

print(f"Interaction Effect Analysis:")
print(f"  Current w[M·T] = {w_MT_base:.8f}")
print(f"  Optimal w[M·T] = {w_MT_optimal:.8f}")
print(f"  Loss at current: {cost_values[np.argmin(np.abs(w_MT_range - w_MT_base))]:.8f}")
print(f"  Loss at optimal: {J_min:.8f}")

In [ ]:
# Plot cost vs interaction coefficient
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(w_MT_range * 1e5, cost_values, linewidth=2.5, color='blue')
plt.scatter([w_MT_optimal * 1e5], [J_min], color='red', s=150, marker='*', 
            label=f'Optimal (w={w_MT_optimal:.2e})', zorder=5)
plt.scatter([w_MT_base * 1e5], [cost_values[np.argmin(np.abs(w_MT_range - w_MT_base))]], 
            color='green', s=100, marker='o', label=f'Trained (w={w_MT_base:.2e})', zorder=5)
plt.xlabel('M·T Coefficient (×10⁻⁵)', fontsize=11, fontweight='bold')
plt.ylabel('Cost J(w, b)', fontsize=11, fontweight='bold')
plt.title('Cost Function vs Interaction Coefficient', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# Zoom in around the minimum
zoom_range = 5  # indices around minimum
start_idx = max(0, min_idx - zoom_range)
end_idx = min(len(w_MT_range), min_idx + zoom_range + 1)
plt.plot(w_MT_range[start_idx:end_idx] * 1e5, cost_values[start_idx:end_idx], 
         'o-', linewidth=2.5, markersize=8, color='blue')
plt.scatter([w_MT_optimal * 1e5], [J_min], color='red', s=150, marker='*', 
            label='Minimum', zorder=5)
plt.xlabel('M·T Coefficient (×10⁻⁵)', fontsize=11, fontweight='bold')
plt.ylabel('Cost J(w, b)', fontsize=11, fontweight='bold')
plt.title('Cost Function (Zoomed)', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Interaction Effect Interpretation ===")
print(f"The parabolic shape indicates a clear optimal value for the M·T interaction term.")
print(f"Both very small and very large w[M·T] increase the cost.")
print(f"The trained model's w[M·T] = {w_MT_base:.2e} is close to optimal,")
print(f"suggesting the gradient descent found a good solution.")
print(f"\nThe interaction term M·T is IMPORTANT for capturing the combined")
print(f"effect of mass and temperature on luminosity.")

## 8. Inference: Predict Luminosity for New Stars

In [ ]:
# Test cases for inference
test_stars = [
    {'name': 'Test Star 1', 'M': 1.3, 'T': 6600},
    {'name': 'Sun-like', 'M': 1.0, 'T': 5800},
    {'name': 'Massive star', 'M': 2.5, 'T': 10000},
]

print("="*90)
print("INFERENCE: Predicting Luminosity for New Stars")
print("="*90)

for star in test_stars:
    M_new = star['M']
    T_new = star['T']
    
    print(f"\n{star['name']}:")
    print(f"  Input: M = {M_new:.1f} M☉, T = {T_new:.0f} K")
    
    # Predict with all three models
    for label in models.keys():
        X = models[label]['X']
        w = models[label]['w']
        b = models[label]['b']
        
        # Build feature vector
        if label == 'M1: [M, T]':
            x_new = np.array([M_new, T_new])
        elif label == 'M2: [M, T, M²]':
            x_new = np.array([M_new, T_new, M_new**2])
        elif label == 'M3: [M, T, M², M·T]':
            x_new = np.array([M_new, T_new, M_new**2, M_new*T_new])
        
        # Predict
        L_pred = x_new @ w + b
        print(f"  {label}: L_pred = {L_pred:.4f} L☉")

print("\n" + "="*90)
print("INTERPRETATIONS:")
print("="*90)
print("\n1. All three models predict different values due to different feature sets.")
print("\n2. M3 (full model) typically gives the most realistic predictions because:")
print("   - It captures nonlinear mass dependence (M²)")
print("   - It includes temperature effects")
print("   - It accounts for M-T interaction")
print("\n3. Predictions for test stars:")
print("   - Should fall within or near the training data range for reliability")
print("   - Extrapolations beyond the training range may be less accurate")

# Check data range
print(f"\nTraining data ranges:")
print(f"  M: {M.min():.1f} - {M.max():.1f} M☉")
print(f"  T: {T.min():.0f} - {T.max():.0f} K")
print(f"  L: {L.min():.2f} - {L.max():.1f} L☉")

## 9. Final Summary and Conclusions

In [ ]:
print("="*90)
print("POLYNOMIAL REGRESSION SUMMARY")
print("="*90)

print("\n1. FEATURE ENGINEERING")
print("-" * 90)
print("   We constructed three feature matrices:")
print("   - M1 (linear): [M, T] - captures basic linear relationships")
print("   - M2 (quadratic): [M, T, M²] - adds nonlinear mass effect")
print("   - M3 (full): [M, T, M², M·T] - includes interaction term")

print("\n2. MODEL PERFORMANCE COMPARISON")
print("-" * 90)
for label in models.keys():
    X = models[label]['X']
    w = models[label]['w']
    b = models[label]['b']
    L_hat = predict_poly(X, w, b)
    residuals = L_hat - L
    mse = np.mean(residuals**2)
    rmse = np.sqrt(mse)
    r2 = 1 - np.sum(residuals**2) / np.sum((L - np.mean(L))**2)
    
    print(f"\n   {label}:")
    print(f"     MSE = {mse:.8f}")
    print(f"     RMSE = {rmse:.6f}")
    print(f"     R² = {r2:.6f}")
    print(f"     Parameters: w = {w}")
    print(f"                 b = {b:.8f}")

print("\n3. KEY FINDINGS")
print("-" * 90)
print("   • M3 (full model) provides the BEST fit (lowest loss)")
print("   • The M² term is CRUCIAL for capturing rapid L increase at high M")
print("   • The M·T interaction term is IMPORTANT for accurate predictions")
print("   • Temperature (T) contributes independently to luminosity")

print("\n4. COMPARISON WITH PART 1 (Linear Model)")
print("-" * 90)
print("   Part 1 (linear M only): L = w·M + b")
print("   - Simple but LIMITED")
print("   - Ignores temperature")
print("   - Cannot capture nonlinearity")
print("\n   Part 2 (polynomial, M+T+M²+M·T):")
print("   - Much IMPROVED predictions")
print("   - Captures astrophysical complexity")
print("   - Significantly lower residuals")

print("\n5. PHYSICAL INTERPRETATION")
print("-" * 90)
print("   The Stefan-Boltzmann law: L ∝ R²·T⁴")
print("   For main-sequence stars: R ∝ M (hydrostatic equilibrium)")
print("   Therefore: L ∝ M²·T⁴ (approximately)")
print(f"\n   Our learned interaction term (w[M·T] = {w_full[3]:.2e}) captures")
print("   this coupling between M and T, though the exponents differ due to")
print("   the simplified linear feature space.")

print("\n" + "="*90)